# LAB 01 — Part D: Inspect the Sparse Representation

Notebook này thực hiện đúng pipeline: **Raw documents → Tokenizer → Count → TF → IDF → TF-IDF**.

Không sử dụng `sklearn.TfidfVectorizer`. Corpus được lưu dưới dạng sparse dictionary để không tạo ma trận dense kích thước rất lớn.

## 1. Cài đặt và tìm dataset

Cell dưới đây chạy được trên local và Google Colab. Trên Colab, nếu dataset chưa có, notebook sẽ clone repository GitHub của bài.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import re
import subprocess
import sys

DATA_FILENAME = 'c4-train.00000-of-01024-30K.json'
REPO_URL = 'https://github.com/NguyenDucThang-tb/NLP.git'

candidate_paths = [
    Path('/home/nguyenducthang/NLP/Lab1') / DATA_FILENAME,
    Path('/content/NLP/Lab1') / DATA_FILENAME,
    Path.cwd() / DATA_FILENAME,
    Path.cwd() / 'Lab1' / DATA_FILENAME,
]
DATA_PATH = next((p for p in candidate_paths if p.is_file()), None)

if DATA_PATH is None and Path('/content').exists():
    repo_dir = Path('/content/NLP')
    if not (repo_dir / '.git').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    matches = list(repo_dir.rglob(DATA_FILENAME))
    DATA_PATH = matches[0] if matches else None

if DATA_PATH is None:
    raise FileNotFoundError(
        f'Không tìm thấy {DATA_FILENAME}. Đặt dataset vào thư mục Lab1 hoặc /content/NLP/Lab1.'
    )

print('Dataset:', DATA_PATH)

## 2. Dataset và tokenizer

Mỗi dòng của file là một JSON object có trường `text`. Các dòng không hợp lệ hoặc document rỗng sẽ được bỏ qua và đếm lại.

In [ ]:
TOKEN_RE = re.compile(r'(?u)\b\w+\b')

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

raw_count = 0
invalid_count = 0
empty_count = 0
documents = []

with DATA_PATH.open('r', encoding='utf-8') as file:
    for line in file:
        raw_count += 1
        try:
            item = json.loads(line)
            text = item.get('text', '') if isinstance(item, dict) else ''
        except (json.JSONDecodeError, UnicodeDecodeError):
            invalid_count += 1
            continue
        if not tokenize(text):
            empty_count += 1
            continue
        documents.append({'text': text, 'url': item.get('url', '')})

texts = [doc['text'] for doc in documents]
N = len(texts)
token_lists = [tokenize(text) for text in texts]
token_lengths = [len(tokens) for tokens in token_lists]

print(f'Raw documents = {raw_count:,}')
print(f'Valid documents = {N:,}')
print(f'Invalid JSON lines = {invalid_count:,}')
print(f'Empty documents = {empty_count:,}')
print(f'Average tokens/document = {sum(token_lengths) / N:.2f}')
print(f'Min tokens/document = {min(token_lengths)}')
print(f'Max tokens/document = {max(token_lengths)}')

## 3. Xây dựng Count, TF, IDF và TF-IDF sparse

Với document `d` và term `t`:

- `TF(t,d) = count(t,d) / tổng số token trong d`
- `IDF(t) = log(N / df(t))` theo công thức của đề
- `TF-IDF(t,d) = TF(t,d) × IDF(t)`

Mỗi document được lưu dưới dạng `{term: value}` và chỉ giữ các giá trị khác 0. Đây chính là biểu diễn sparse của ma trận.

In [ ]:
# Vocabulary: sắp xếp để thứ tự cột luôn tái lập được
vocabulary = {term: index for index, term in enumerate(sorted({t for tokens in token_lists for t in tokens}))}
V = len(vocabulary)

# Count representation: mỗi dòng là một Counter sparse
count_rows = [Counter(tokens) for tokens in token_lists]

# Document frequency df(t)
df = Counter()
for row in count_rows:
    df.update(row.keys())

idf = {term: math.log(N / df[term]) for term in vocabulary}

tfidf_rows = []
for tokens, counts in zip(token_lists, count_rows):
    length = len(tokens)
    row = {}
    for term, count in counts.items():
        value = (count / length) * idf[term]
        if value != 0.0:
            row[term] = value
    tfidf_rows.append(row)

nnz = sum(len(row) for row in tfidf_rows)
total_elements = N * V
sparsity = 1 - nnz / total_elements
density = nnz / total_elements

assert N > 0 and V > 0
assert 0 <= density <= 1 and 0 <= sparsity <= 1
print('Count/TF-IDF sparse representation created successfully.')

## 4. Kích thước ma trận và sparsity

Nếu `X ∈ R^(N×V)` thì:

- `N` là số document.
- `V` là số term khác nhau trong vocabulary.
- `X.shape = (N, V)`.

Sparsity được tính bằng `S = 1 - nnz(X)/(N×V)`.

In [ ]:
print(f'Number of documents (N) = {N:,}')
print(f'Vocabulary size (V) = {V:,}')
print(f'Matrix shape = ({N:,}, {V:,})')
print(f'nnz(X) = {nnz:,}')
print(f'Total matrix elements = {total_elements:,}')
print(f'Sparsity = {sparsity:.6f} ({sparsity * 100:.4f}%)')
print(f'Density = {density:.6f} ({density * 100:.4f}%)')

### Trả lời câu hỏi về sparsity

Một document chỉ chứa một phần rất nhỏ vocabulary nên chỉ có một số ít phần tử khác 0. Tuy nhiên, tất cả document phải dùng cùng một vocabulary chung để các cột có cùng ý nghĩa giữa các document. Vì vậy mỗi vector vẫn có `V` chiều; phần lớn các chiều không xuất hiện trong document đó và được lưu bằng 0. Đây là lý do dùng sparse representation giúp tiết kiệm bộ nhớ.

## 5. Inspect vocabulary

Ba danh sách dưới đây được tính trực tiếp từ corpus, không hard-code kết quả:

1. 20 term có document frequency cao nhất.
2. 20 term có IDF cao nhất.
3. 20 term có TF-IDF cao nhất trong document được chọn.

In [ ]:
def print_table(title, rows, headers):
    print('\n' + title)
    print(' | '.join(headers))
    print('-' * 80)
    for row in rows:
        print(' | '.join(str(value) for value in row))

top_df = sorted(((term, df[term], df[term] / N) for term in vocabulary), key=lambda x: (-x[1], x[0]))[:20]
print_table('Top 20 terms by document frequency', [(t, f'{freq:,}', f'{ratio:.6f}') for t, freq, ratio in top_df], ['term', 'df', 'df/N'])

top_idf = sorted(((term, idf[term], df[term]) for term in vocabulary), key=lambda x: (-x[1], x[0]))[:20]
print_table('Top 20 terms by IDF', [(t, f'{value:.6f}', freq) for t, value, freq in top_idf], ['term', 'idf', 'df'])

chosen_doc_index = 0
chosen_doc_tfidf = sorted(tfidf_rows[chosen_doc_index].items(), key=lambda x: (-x[1], x[0]))[:20]
print(f'\nSelected document index = {chosen_doc_index}')
print('URL =', documents[chosen_doc_index]['url'])
print('Text preview =', texts[chosen_doc_index][:300].replace('\n', ' '))
print_table('Top 20 terms by TF-IDF in selected document', [(t, f'{value:.8f}', df[t], f'{idf[t]:.6f}') for t, value in chosen_doc_tfidf], ['term', 'tfidf', 'df', 'idf'])

## 6. So sánh và nhận xét

- Terms có **DF cao** xuất hiện trong nhiều document, thường là các từ chung; vì vậy IDF của chúng thường thấp.
- Terms có **IDF cao** xuất hiện trong rất ít document, nên có khả năng phân biệt document tốt hơn.
- Terms có **TF-IDF cao trong một document** vừa phải xuất hiện đủ nhiều trong document đó, vừa không xuất hiện quá phổ biến trong toàn corpus.
- Do đó ba danh sách không nhất thiết giống nhau: DF đo độ phổ biến toàn corpus, IDF đo độ hiếm toàn corpus, còn TF-IDF kết hợp độ quan trọng trong một document cụ thể.

Hãy dùng các số liệu và ba bảng được in ở trên để bổ sung nhận xét cụ thể của bạn vào báo cáo.

# Part F — Experiment 2: Preprocessing Ablation

Mục tiêu của experiment là kiểm chứng bằng số liệu xem mỗi lựa chọn preprocessing ảnh hưởng thế nào đến representation và search. Ba pipeline dùng cùng corpus và cùng các query, chỉ thay đổi cách tạo token.

## Định nghĩa ba pipeline

- **Pipeline A — Minimal:** lowercase → tokenization.
- **Pipeline B — Normalized:** lowercase → thay punctuation bằng khoảng trắng → tokenization → loại một bộ stopword tiếng Anh nhỏ, xác định rõ trong code.
- **Pipeline C — Extended:** normalization → giữ word token và thêm prefix/suffix subword units.

Pipeline C là một cách mô phỏng subword tokenization bằng prefix/suffix units, dùng thư viện chuẩn Python và không dùng sklearn.

In [ ]:
import gc

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from',
    'has', 'have', 'he', 'her', 'his', 'i', 'in', 'is', 'it', 'its',
    'of', 'on', 'or', 'that', 'the', 'their', 'there', 'they', 'this',
    'to', 'was', 'we', 'were', 'will', 'with', 'you', 'your'
}

def minimal_tokens(text):
    return TOKEN_RE.findall(text.lower())

def normalized_words(text, remove_stopwords=True):
    cleaned = re.sub(r'[^\w\s]', ' ', text.lower())
    words = TOKEN_RE.findall(cleaned)
    if remove_stopwords:
        words = [word for word in words if word not in STOPWORDS]
    return words

def extended_tokens(text):
    words = normalized_words(text, remove_stopwords=False)
    subwords = []
    for word in words:
        if len(word) >= 3:
            subwords.append('sub_prefix:' + word[:3])
            subwords.append('sub_suffix:' + word[-3:])
    return words + subwords

PIPELINES = {
    'A Minimal': minimal_tokens,
    'B Normalized': normalized_words,
    'C Extended': extended_tokens,
}

QUERIES = ['classification', 'machine learning', 'healthcare', 'natural language processing', 'data model']
print('Queries used for every pipeline:', QUERIES)

## Hàm fit sparse và search

Các hàm dưới đây tạo vocabulary, TF-IDF sparse và tìm document bằng cosine similarity. mean_top1_cosine_proxy chỉ là search signal không có nhãn; để kết luận search pipeline nào tốt nhất một cách nghiêm ngặt cần thêm relevance labels và các metric như P@5, Recall@5 hoặc MRR.

In [ ]:
def fit_sparse(tokenizer):
    # Streaming build: avoid keeping a second list containing every token string.
    count_rows = []
    token_lengths = []
    document_frequency = Counter()
    vocabulary_terms = set()
    for text in texts:
        tokens = tokenizer(text)
        counts = Counter(tokens)
        count_rows.append(counts)
        token_lengths.append(len(tokens))
        vocabulary_terms.update(counts.keys())
        document_frequency.update(counts.keys())
    vocab = {term: index for index, term in enumerate(sorted(vocabulary_terms))}
    idf_values = {term: math.log(N / document_frequency[term]) for term in vocab}
    document_norms = []
    nnz_value = 0
    for length, counts in zip(token_lengths, count_rows):
        squared_norm = sum((count / length * idf_values[term]) ** 2 for term, count in counts.items() if length and idf_values[term] != 0.0)
        document_norms.append(math.sqrt(squared_norm))
        nnz_value += sum(idf_values[term] != 0.0 for term in counts)
    return token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency, nnz_value

def query_vector(query, tokenizer, vocab, idf_values):
    tokens = tokenizer(query)
    counts = Counter(tokens)
    total = len(tokens)
    vector = {term: count / total * idf_values[term] for term, count in counts.items() if total and term in vocab and idf_values[term] != 0.0}
    return vector, tokens

def sparse_cosine(x, y, norm_x=None, norm_y=None):
    if norm_x is None:
        norm_x = math.sqrt(sum(value * value for value in x.values()))
    if norm_y is None:
        norm_y = math.sqrt(sum(value * value for value in y.values()))
    if not norm_x or not norm_y:
        return 0.0
    smaller, larger = (x, y) if len(x) <= len(y) else (y, x)
    dot = sum(value * larger.get(term, 0.0) for term, value in smaller.items())
    return dot / (norm_x * norm_y)

def search_top1(query, tokenizer, vocab, idf_values, count_rows, document_norms, token_lengths, k=5):
    q_vector, query_tokens = query_vector(query, tokenizer, vocab, idf_values)
    q_norm = math.sqrt(sum(value * value for value in q_vector.values()))
    scored = []
    for index, counts in enumerate(count_rows):
        length = token_lengths[index]
        dot = sum(q_value * counts.get(term, 0) / length * idf_values[term] for term, q_value in q_vector.items()) if length else 0.0
        d_norm = document_norms[index]
        score = dot / (q_norm * d_norm) if q_norm and d_norm else 0.0
        scored.append((score, index))
    return sorted(scored, key=lambda item: (-item[0], item[1]))[:k], query_tokens

## Đo lường và bảng so sánh

OOV rate được tính trên 5 query chung: số query token không có trong vocabulary chia cho tổng số query token. Search performance ở đây báo cáo mean_top1_cosine_proxy; đây là proxy để quan sát, không phải đánh giá relevance cuối cùng.

In [ ]:
ablation_results = []

for pipeline_name, tokenizer in PIPELINES.items():
    token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency, pipeline_nnz = fit_sparse(tokenizer)
    pipeline_v = len(vocab)
    pipeline_avg_tokens = sum(token_lengths) / N
    pipeline_sparsity = 1 - pipeline_nnz / (N * pipeline_v)
    all_query_tokens = []
    top1_scores = []
    for query in QUERIES:
        ranked, query_tokens = search_top1(query, tokenizer, vocab, idf_values, count_rows, document_norms, token_lengths)
        all_query_tokens.extend(query_tokens)
        top1_scores.append(ranked[0][0] if ranked else 0.0)
    oov_count = sum(token not in vocab for token in all_query_tokens)
    oov_rate = oov_count / len(all_query_tokens) if all_query_tokens else 0.0
    mean_top1 = sum(top1_scores) / len(top1_scores)
    row = {
        'pipeline': pipeline_name,
        'vocabulary_size': pipeline_v,
        'average_tokens_per_document': pipeline_avg_tokens,
        'matrix_sparsity': pipeline_sparsity,
        'oov_rate': oov_rate,
        'mean_top1_cosine_proxy': mean_top1,
    }
    ablation_results.append(row)
    print('\n' + pipeline_name)
    for metric, value in row.items():
        if metric != 'pipeline':
            print(f'  {metric}: {value:.6f}' if isinstance(value, float) else f'  {metric}: {value:,}')
    del token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency
    gc.collect()

print('\nComparison table')
print('Pipeline | Vocabulary | Avg tokens/doc | Sparsity | OOV rate | Mean Top-1 cosine proxy')
print('-' * 105)
for row in ablation_results:
    print(f"{row['pipeline']} | {row['vocabulary_size']:,} | {row['average_tokens_per_document']:.2f} | {row['matrix_sparsity']:.6f} | {row['oov_rate']:.4f} | {row['mean_top1_cosine_proxy']:.6f}")

## Câu hỏi phân tích

Dùng bảng kết quả vừa in để trả lời, không khẳng định preprocessing càng nhiều càng tốt:

1. **Lowercasing làm thay đổi vocabulary thế nào?** So sánh A với B; do A đã lowercase nên thay đổi chính của B đến từ punctuation và stopword handling.
2. **Stopword removal có luôn cải thiện representation không?** Không nhất thiết. Nó có thể giảm vocabulary và số token, nhưng có thể làm mất tín hiệu ngữ nghĩa hoặc thay đổi search.
3. **Loại punctuation có thể mất thông tin gì?** Dấu câu có thể biểu thị cảm xúc, phủ định, cấu trúc câu, số thập phân, mã phiên bản hoặc ký hiệu kỹ thuật.
4. **Pipeline nào sparse nhất?** Chọn pipeline có matrix_sparsity lớn nhất trong bảng.
5. **Pipeline nào search tốt nhất?** Chỉ được kết luận chắc chắn sau khi có relevance labels và P@5/Recall@5/MRR. Nếu chỉ dùng kết quả hiện tại, hãy gọi pipeline có mean_top1_cosine_proxy cao nhất là pipeline có proxy score cao nhất.
6. **Search tốt hơn có đồng nghĩa vocabulary nhỏ hơn không?** Không. Vocabulary size và chất lượng search là hai thuộc tính khác nhau; hãy đối chiếu trực tiếp hai cột trong bảng.

# Part G — Application: Build a Document Search Engine

Search engine sử dụng Pipeline B — Normalized từ Part F làm index. Quy trình là: **documents → TF-IDF index → query TF-IDF vector → cosine similarity → ranking → Top-5 documents**.

## Query examples

Các query được dùng theo đúng đề bài:

- medical image classification
- transformer language model
- deep learning healthcare
- natural language processing

In [ ]:
# Fit lại Pipeline B để xây dựng search index sau khi Part F đã hoàn tất.
search_token_lengths, search_vocab, search_idf, search_count_rows, search_doc_norms, search_df, search_nnz = fit_sparse(normalized_words)
SEARCH_QUERIES = [
    'medical image classification',
    'transformer language model',
    'deep learning healthcare',
    'natural language processing',
]

print(f'Search index documents = {N:,}')
print(f'Search index vocabulary = {len(search_vocab):,}')
print(f'Search index sparsity = {1 - search_nnz / (N * len(search_vocab)):.6f}')

In [ ]:
def print_search_results(query, ranked, query_tokens):
    print('\n' + '=' * 120)
    print('QUERY:', query)
    known_tokens = [token for token in query_tokens if token in search_vocab]
    unknown_tokens = [token for token in query_tokens if token not in search_vocab]
    print('Query tokens:', query_tokens)
    print('Known tokens:', known_tokens if known_tokens else 'none')
    print('OOV tokens:', unknown_tokens if unknown_tokens else 'none')
    if not known_tokens:
        print('WARNING: query has no token in the vocabulary; all similarities are 0.')
    print('\nRank | Document ID | Similarity | Document preview | URL')
    print('-' * 120)
    for rank, (score, row_index) in enumerate(ranked, start=1):
        preview = ' '.join(texts[row_index].split())[:200]
        url = documents[row_index].get('url', '')
        print(f'{rank:>4} | {row_index:>11} | {score:>10.6f} | {preview} | {url}')


search_output_rows = []
for query in SEARCH_QUERIES:
    ranked, query_tokens = search_top1(
        query,
        normalized_words,
        search_vocab,
        search_idf,
        search_count_rows,
        search_doc_norms,
        search_token_lengths,
        k=5,
    )
    print_search_results(query, ranked, query_tokens)
    for rank, (score, row_index) in enumerate(ranked, start=1):
        search_output_rows.append({
            'query': query,
            'rank': rank,
            'document_id': row_index,
            'similarity': score,
            'url': documents[row_index].get('url', ''),
            'preview': ' '.join(texts[row_index].split())[:200],
        })

print(f'\nSaved in-memory search rows: {len(search_output_rows)}')

## Observation guide for Part G

Sau khi chạy, hãy đọc preview và URL của từng Top-5 để tự nhận xét:

1. Query nào trả về các document có nội dung phù hợp nhất?
2. Query nào có similarity cao hoặc thấp bất thường?
3. Có document nào chứa đúng nhiều query terms nhưng vẫn không đứng đầu không? Có thể do TF-IDF giảm trọng số các term quá phổ biến.
4. Nếu query có OOV token, hệ thống vẫn chạy; token đó bị bỏ qua khi tạo query vector.
5. Nếu toàn bộ query là OOV, cosine similarity của mọi document bằng 0 và kết quả chỉ là tie-break theo document ID.

Similarity cao chỉ cho biết vector gần query hơn; chưa đủ để kết luận document thực sự relevant. Muốn đánh giá search quality cần thêm relevance labels ở Part H.

# Part H — Evaluation

Part H đánh giá search engine bằng một evaluation set nhỏ có relevance labels. Không cần gán nhãn toàn bộ 30K documents; chỉ cần xác định các document relevant cho từng query.

## 1. Tạo evaluation set và điền relevance labels

Notebook dùng 5 query. Document ID chính là chỉ số dòng của document trong corpus, được in ở Part G. Bạn cần tự đọc preview/URL của Top-5 rồi điền các ID relevant vào dictionary dưới đây.

Ví dụ: RELEVANCE_LABELS['medical image classification'] = {12, 45, 78}. Không điền document chỉ vì nó đứng Top-5; chỉ điền khi bạn kiểm tra nội dung và thấy relevant.

In [ ]:
EVAL_QUERIES = SEARCH_QUERIES + ['data model']

# TODO: tự điền document IDs relevant sau khi đọc kết quả Part G.
# Các set rỗng là placeholder, không phải kết quả đánh giá.
RELEVANCE_LABELS = {
    'medical image classification': set(),
    'transformer language model': set(),
    'deep learning healthcare': set(),
    'natural language processing': set(),
    'data model': set(),
}

assert set(EVAL_QUERIES) == set(RELEVANCE_LABELS)
assert all(isinstance(ids, set) for ids in RELEVANCE_LABELS.values())
print('Evaluation queries:', len(EVAL_QUERIES))
print('Labels filled:', sum(bool(ids) for ids in RELEVANCE_LABELS.values()), '/', len(EVAL_QUERIES))

## 2. Precision@5, Recall@5 và Reciprocal Rank

Với mỗi query, hệ thống lấy Top-5 document từ cùng search index của Part G:

- P@5 = số document relevant trong Top-5 / 5.
- Recall@5 = số document relevant trong Top-5 / tổng số document relevant đã gán nhãn.
- RR = 1 / rank của document relevant đầu tiên; nếu không có relevant document trong Top-5 thì RR = 0.
- MRR là trung bình RR trên các query có labels.

Nếu labels còn rỗng, code chỉ báo pending và không tự tạo metric giả.

In [ ]:
def evaluate_one_query(query, relevant_ids, k=5):
    ranked, query_tokens = search_top1(
        query,
        normalized_words,
        search_vocab,
        search_idf,
        search_count_rows,
        search_doc_norms,
        search_token_lengths,
        k=k,
    )
    retrieved_ids = [row_index for score, row_index in ranked]
    relevant_ids = set(relevant_ids)
    hits = [doc_id for doc_id in retrieved_ids if doc_id in relevant_ids]
    precision_at_k = len(hits) / k
    recall_at_k = len(hits) / len(relevant_ids) if relevant_ids else None
    reciprocal_rank = next((1 / rank for rank, doc_id in enumerate(retrieved_ids, start=1) if doc_id in relevant_ids), 0.0)
    assert 0.0 <= precision_at_k <= 1.0
    assert 0.0 <= reciprocal_rank <= 1.0
    if recall_at_k is not None:
        assert 0.0 <= recall_at_k <= 1.0
    return {
        'query': query,
        'retrieved_ids': retrieved_ids,
        'relevant_ids': sorted(relevant_ids),
        'hits': hits,
        'precision_at_5': precision_at_k,
        'recall_at_5': recall_at_k,
        'reciprocal_rank': reciprocal_rank,
    }


evaluation_rows = []
for query in EVAL_QUERIES:
    result = evaluate_one_query(query, RELEVANCE_LABELS[query], k=5)
    evaluation_rows.append(result)
    if result['relevant_ids']:
        print(f"{query}: P@5={result['precision_at_5']:.4f}, Recall@5={result['recall_at_5']:.4f}, RR={result['reciprocal_rank']:.4f}, hits={result['hits']}")
    else:
        print(f'{query}: pending — chưa có relevance labels')

labeled_rows = [row for row in evaluation_rows if row['relevant_ids']]
if labeled_rows:
    mean_precision = sum(row['precision_at_5'] for row in labeled_rows) / len(labeled_rows)
    mean_recall = sum(row['recall_at_5'] for row in labeled_rows) / len(labeled_rows)
    mrr = sum(row['reciprocal_rank'] for row in labeled_rows) / len(labeled_rows)
    print('\nMean metrics over labeled queries')
    print(f'Mean P@5 = {mean_precision:.4f}')
    print(f'Mean Recall@5 = {mean_recall:.4f}')
    print(f'MRR = {mrr:.4f}')
else:
    mean_precision = mean_recall = mrr = None
    print('\nEvaluation is pending: hãy điền RELEVANCE_LABELS rồi chạy lại cell này.')

## 3. Interpretation

Sau khi điền labels, báo cáo ba metric chính: mean P@5, mean Recall@5 và MRR. P@5 cao nghĩa là Top-5 chứa nhiều document relevant; Recall@5 cao nghĩa là Top-5 tìm được nhiều trong số các document relevant đã biết; MRR cao nghĩa là document relevant đầu tiên thường xuất hiện ở rank cao.

Nếu một query có Recall@5 thấp, không được kết luận ngay rằng preprocessing hoặc TF-IDF sai. Hãy kiểm tra thêm: relevance labels có đầy đủ không, query có OOV không, document relevant có thật sự nằm trong corpus không, và từ khóa query có bị stopword handling loại bỏ không.